[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mauriciorslrv/DS_basics/blob/main/03%20Aprendizaje%20M%C3%A1quina/notebooks/05_Clasificacion_Bank_Marketing.ipynb)

# 05 · Clasificación: Bank Marketing

> **Pregunta:** antes de llamar, ¿podemos priorizar a quién contactar para ofrecer un depósito a plazo?

Usaremos **Bank Marketing**, conjunto público de campañas telefónicas de una institución bancaria portuguesa, publicado por Moro, Cortez y Rita en UCI. Cada fila representa un contacto durante la campaña; **y** indica si la persona suscribió el depósito (**yes/no**). Es clasificación binaria.

Como predecimos antes de llamar, excluimos **duration**, conocida al terminar la llamada, y **campaign**, que resume el número total de contactos realizados durante la campaña. La predicción no prueba que llamar cause una suscripción.

📊 [UCI Bank Marketing · variables, atribución y licencia CC BY 4.0](https://archive.ics.uci.edu/dataset/222/bank+marketing) · DOI 10.24432/C5K306.

## 1. Cargar y conocer los datos

Descargamos los datos reales de UCI y comprobamos tamaño, tipos y proporción de cada etiqueta. La ficha de la fuente define campos demográficos, de campaña, de contactos previos y de contexto económico; consultarla evita inferir significado sólo por el nombre abreviado.

In [ ]:
# La etiqueta de UCI registra suscripción sí/no; las variables se limitan al escenario antes de una llamada.
%pip -q install ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

bank = fetch_ucirepo(id=222)
X_original = bank.data.features.copy()
y_original = bank.data.targets.copy()
df = X_original.copy()
df["y"] = y_original.iloc[:, 0].astype(str).str.lower().to_numpy()

print("Dimensiones:", df.shape)
print("Target:", df["y"].value_counts(dropna=False))
print("Tasa positiva:", round((df["y"] == "yes").mean(), 3))
display(df.head())
display(df.dtypes.to_frame("tipo"))

### ¿Está desbalanceada la etiqueta?

Si la mayoría de las personas no suscribe, una regla que siempre predice “no” puede tener accuracy alta y aun así no encontrar a quienes sí suscriben.

In [ ]:
df["y"].value_counts(normalize=True).plot(kind="bar", color=["#176b87", "#b64a36"])
plt.title("Proporción de suscripción")
plt.ylabel("Proporción")
plt.xticks(rotation=0)
plt.show()

## 2. Definir el escenario y evitar fuga

La pregunta es a quién llamar **antes de marcar**. Conservamos atributos del cliente, el contexto económico y antecedentes de contactos que estarían disponibles antes de una nueva llamada. Las categorías, como empleo o educación, se codifican sin inventar un orden numérico.

Excluimos **duration** porque es la duración de la llamada actual, conocida sólo después de que sucede. Excluimos **campaign** porque cuenta los contactos de esta campaña y su valor final puede incluir contactos posteriores a la decisión de priorizar. Si la pregunta fuera describir resultados al cierre, las columnas podrían ser distintas; aquí buscamos una decisión previa al contacto.

In [ ]:
y = (df["y"] == "yes").astype(int)
X = df.drop(columns=["y", "duration", "campaign"], errors="ignore").copy()

print("Se excluyó duration.")
print("Variables de entrada:", X.shape[1])
print("Tasa de suscripción:", round(y.mean(), 3))
print("Campos tipo texto:", X.select_dtypes(include=["object", "category"]).columns.tolist())

## 3. Interpretar pdays=999

UCI define **pdays = 999** como que no hubo contacto previo; no son literalmente 999 días. Creamos **not_previously_contacted** para conservar ese significado y cambiamos el centinela a faltante. El imputador numérico aprende un valor sólo desde entrenamiento; la bandera mantiene explícita la información de que no hubo contacto anterior.

In [ ]:
if "pdays" in X.columns:
    X["not_previously_contacted"] = (X["pdays"] == 999).astype(int)
    X["pdays"] = X["pdays"].replace(999, np.nan)

unknown_counts = {
    col: int((X[col].astype(str).str.lower() == "unknown").sum())
    for col in X.select_dtypes(include=["object", "category"]).columns
}
display(pd.Series(unknown_counts, name="unknown por columna").sort_values(ascending=False).head(10))

## 4. Separar entrenamiento y prueba

Usamos partición estratificada para mantener proporciones de clase parecidas y poder evaluar con ambas clases presentes. La semilla fija hace reproducible la división, pero no garantiza que otro periodo o banco se comporte igual. En la notebook 06 reservamos además validación para elegir modelo y umbral sin consultar el test final.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Tasa positiva train:", round(y_train.mean(), 3))
print("Tasa positiva test:", round(y_test.mean(), 3))
print("Tamaños:", X_train.shape, X_test.shape)

## 5. Preparar variables dentro del pipeline

Las columnas numéricas se imputan y escalan; las categóricas se imputan y codifican con one-hot, que crea indicadores sin imponerles orden. El pipeline aprende estas transformaciones con entrenamiento y las aplica a prueba sin recalcular parámetros, reduciendo la fuga en la preparación.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

categorical = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric = [c for c in X_train.columns if c not in categorical]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])

model = Pipeline([
    ("prep", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])
model.fit(X_train, y_train)
print("Modelo ajustado.")

## 6. Comparar con una regla simple

El baseline siempre predice la clase más frecuente. Sirve para saber si el modelo aprende algo útil frente a una regla ingenua. Como “no suscribe” es más común, accuracy puede ser alta aunque no identifique suscriptores. **Precision** indica qué proporción de priorizados suscribió; **recall**, qué proporción de suscriptores encontramos; **F1** resume el balance entre ambas.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
pred_base = baseline.predict(X_test)

prob_yes = model.predict_proba(X_test)[:, 1]
pred_50 = (prob_yes >= 0.50).astype(int)

print("Accuracy baseline:", round(accuracy_score(y_test, pred_base), 3))
print("\nReporte de clasificación, umbral 0.50\n")
print(classification_report(
    y_test, pred_50, target_names=["no suscribe", "suscribe"], zero_division=0
))

## 7. Matriz de confusión y cambio de umbral

Un falso positivo prioriza a alguien que no suscribe; un falso negativo deja fuera a alguien que sí habría suscrito. El costo depende del propósito y la capacidad del equipo, datos que UCI no proporciona.

Aquí comparamos umbrales sólo para enseñar cómo cambian las métricas. Esta celda usa la partición de prueba, así que sus resultados **no deben usarse para elegir el umbral final**. La notebook 06 muestra el flujo correcto: elegir con validación y consultar el test reservado al final.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, pred_50, display_labels=["No", "Sí"], cmap="Blues", values_format="d"
)
plt.title("Test · umbral 0.50")
plt.show()

from sklearn.metrics import precision_score, recall_score, f1_score
rows = []
for threshold in [0.30, 0.50, 0.70]:
    predicted = (prob_yes >= threshold).astype(int)
    rows.append({
        "umbral": threshold,
        "precision": precision_score(y_test, predicted, zero_division=0),
        "recall": recall_score(y_test, predicted, zero_division=0),
        "F1": f1_score(y_test, predicted, zero_division=0)
    })
pd.DataFrame(rows).set_index("umbral").round(3)

## 🧠 Interpretación responsable

La probabilidad resume patrones históricos de esta institución y campaña; no garantiza un resultado individual ni estima causalmente el efecto de llamar. No basta para decidir automáticamente a quién contactar: hay que revisar objetivo, costos, privacidad, sesgos y desempeño con datos actuales.

## 🧪 Mini reto

Compara precision y recall con umbrales 0.30, 0.50 y 0.70 sólo como experimento. Si el equipo pudiera realizar pocas llamadas, define una política y justifica qué error sería más costoso. Para elegirla correctamente, diseña una partición de validación y deja el test intacto.

## Fuentes y atribución

- Moro, S., Cortez, P. & Rita, P. (2014). *A data-driven approach to predict the success of bank telemarketing*. UCI Bank Marketing. [Ficha, variables, atribución y licencia CC BY 4.0](https://archive.ics.uci.edu/dataset/222/bank+marketing), DOI 10.24432/C5K306.
- Scikit-learn: [métricas de clasificación](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics) y [pipelines](https://scikit-learn.org/stable/modules/compose.html).